# C2 Assignment Notebook
Replicates and debugs the C2 dispatch pipeline from `src/components/c2.py` and `src/policies/policies.py`.

Pipeline:
```
pending_requests
  → fac_eligibility       (filter destinations per issuing facility — lever_fac_elig)
  → amb_eligibility       (filter ambulances per issuing facility  — lever_amb_elig)
  → solve_ilp             (4-term weighted ILP: α·triage + β·sojourn − γ·transport − δ·EWT)
  → craft_plans           (build pickup/dropoff steps per ambulance)
  → dispatch_ambulances   (mark requests assigned, free ambulances)
```

## Imports

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from pulp import LpProblem, LpVariable, LpMaximize, lpSum, value, PULP_CBC_CMD

## Toy Data

We build minimal stub objects that mirror the real simulation objects only where the policy functions need them.
No `salabim` required — we stub `env`, `Ambulance`, `Facility`, `Patient`, and `TransportRequest`.

In [ ]:
# ── Stubs ─────────────────────────────────────────────────────────────────────

@dataclass
class Patient:
    patient_id: str
    triage: str


@dataclass
class Unit:
    """Stub for a treatment unit inside a Facility."""
    _wait: float = 0.0

    def expected_wait_if_join_now(self) -> float:
        return self._wait


@dataclass
class Facility:
    loc_id: str
    units: Dict[str, Unit] = field(default_factory=dict)
    exit_queue: list = field(default_factory=list)  # list[TransportRequest]


@dataclass
class Ambulance:
    amb_id: str
    capacity: int
    location: "Facility"
    plan: list = field(default_factory=list)

    def _loc_id(self, loc) -> str:
        return loc.loc_id if loc is not None else ""


@dataclass
class PlanStep:
    facility: Facility
    unload_patients: list
    load_patients: list


@dataclass
class TransportRequest:
    req_id: str
    patient: Patient
    issuing_facility: Facility
    t_issued: float
    injury_time: float
    triage: str
    needed_unit: Optional[str]
    status: str = "pending"

    @property
    def casualty_id(self):
        return self.patient.patient_id

    @property
    def issuing_facility_id(self):
        return self.issuing_facility.loc_id


class Env:
    """Minimal env stub — holds the lookup tables the policies need."""

    def __init__(self, now: float = 100.0):
        self._now = now
        self.fac_eligibility_datasets: Dict[int, pd.DataFrame] = {}
        self.amb_eligibility_datasets: Dict[int, pd.DataFrame] = {}
        self.facility_by_id: Dict[str, Facility] = {}
        self.amb_by_id: Dict[str, Ambulance] = {}
        self.distance_matrix: Optional[pd.DataFrame] = None

    def now(self) -> float:
        return self._now

In [ ]:
# ── Scenario ──────────────────────────────────────────────────────────────────
#
# 1 CCP with 10 casualties (3×T1, 4×T2, 3×T3)
# 4 R1 facilities — each with a different distance and expected wait time
#
#   r1_alpha   15 min from ccp,  0 min wait  (closest, currently free)
#   r1_bravo   20 min from ccp, 10 min wait
#   r1_charlie 35 min from ccp,  5 min wait  (far but nearly free)
#   r1_delta   25 min from ccp, 30 min wait  (moderate distance, very busy)
#
# Ambulances (both capacity 2)
#   amb_1  at ccp      (0 min travel to reach casualties)
#   amb_2  at r1_alpha (15 min travel to reach ccp)
#
# Only 4 seats total → 6 patients remain pending each dispatch cycle
# ─────────────────────────────────────────────────────────────────────────────

# Facilities
ccp        = Facility(loc_id="ccp",        units={})
r1_alpha   = Facility(loc_id="r1_alpha",   units={"treatment": Unit( 0.0)})
r1_bravo   = Facility(loc_id="r1_bravo",   units={"treatment": Unit(10.0)})
r1_charlie = Facility(loc_id="r1_charlie", units={"treatment": Unit( 5.0)})
r1_delta   = Facility(loc_id="r1_delta",   units={"treatment": Unit(30.0)})

ALL_FACILITIES = [ccp, r1_alpha, r1_bravo, r1_charlie, r1_delta]

# 10 patients — 3×T1, 4×T2, 3×T3
_triages = ["T1","T1","T1","T2","T2","T2","T2","T3","T3","T3"]
patients = [Patient(f"p{i:03d}", t) for i, t in enumerate(_triages, 1)]

# 10 requests — all at ccp, issued at staggered times (sim now = 100)
# T1 patients were injured/issued earliest; T3 patients most recently
_t_issued = [60, 65, 70, 72, 75, 78, 82, 85, 90, 95]
ALL_REQUESTS = [
    TransportRequest(
        req_id           = f"req_{i:03d}",
        patient          = patients[i - 1],
        issuing_facility = ccp,
        t_issued         = float(_t_issued[i - 1]),
        injury_time      = float(_t_issued[i - 1] - 5),
        triage           = patients[i - 1].triage,
        needed_unit      = "treatment",
    )
    for i in range(1, 11)
]

# Ambulances — both capacity 2
amb_1 = Ambulance(amb_id="amb_1", capacity=2, location=ccp)
amb_2 = Ambulance(amb_id="amb_2", capacity=2, location=r1_alpha)

ALL_AMBULANCES = [amb_1, amb_2]

# ── Distance matrix (travel time in minutes) ─────────────────────────────────
fac_ids = ["ccp", "r1_alpha", "r1_bravo", "r1_charlie", "r1_delta"]
dist_data = [
#           ccp  r1_a  r1_b  r1_c  r1_d
           [  0,   15,   20,   35,   25],   # ccp
           [ 15,    0,   20,   30,   20],   # r1_alpha
           [ 20,   20,    0,   25,   15],   # r1_bravo
           [ 35,   30,   25,    0,   35],   # r1_charlie
           [ 25,   20,   15,   35,    0],   # r1_delta
]
distance_matrix = pd.DataFrame(dist_data, index=fac_ids, columns=fac_ids)

# ── Facility eligibility matrix (lever 0) ─────────────────────────────────────
# rows = issuing facility, cols = eligible destination
# From ccp → any R1 is eligible; R1s are termini (no outbound)
fac_elig_0 = pd.DataFrame(
    [[0, 1, 1, 1, 1],   # from ccp:        all R1s eligible
     [0, 0, 0, 0, 0],   # from r1_alpha:   terminus
     [0, 0, 0, 0, 0],   # from r1_bravo:   terminus
     [0, 0, 0, 0, 0],   # from r1_charlie: terminus
     [0, 0, 0, 0, 0]],  # from r1_delta:   terminus
    index=fac_ids, columns=fac_ids,
)

# ── Ambulance eligibility matrix (lever 0) ────────────────────────────────────
# Both ambulances may serve any facility
amb_elig_0 = pd.DataFrame(
    [[1, 1, 1, 1, 1],   # amb_1
     [1, 1, 1, 1, 1]],  # amb_2
    index=["amb_1", "amb_2"], columns=fac_ids,
)

# ── Build env ─────────────────────────────────────────────────────────────────
env = Env(now=100.0)
env.fac_eligibility_datasets = {0: fac_elig_0}
env.amb_eligibility_datasets = {0: amb_elig_0}
env.distance_matrix          = distance_matrix
env.facility_by_id           = {f.loc_id: f for f in ALL_FACILITIES}
env.amb_by_id                = {a.amb_id: a for a in ALL_AMBULANCES}


def reset_scenario():
    """Restore all request statuses, exit queues, and ambulance plans to initial state."""
    for req in ALL_REQUESTS:
        req.status = "pending"
    ccp.exit_queue = list(ALL_REQUESTS)
    for amb in ALL_AMBULANCES:
        amb.plan = []


reset_scenario()

print(f"Facilities : {[f.loc_id for f in ALL_FACILITIES]}")
print(f"Ambulances : {[a.amb_id for a in ALL_AMBULANCES]}  (capacity {amb_1.capacity} each)")
print(f"Requests   : {len(ALL_REQUESTS)} casualties at ccp")
print(f"Total seats: {sum(a.capacity for a in ALL_AMBULANCES)}  "
      f"→ {len(ALL_REQUESTS) - sum(a.capacity for a in ALL_AMBULANCES)} patients remain pending each cycle")
print()
print("Distance matrix (travel time in minutes):")
print(distance_matrix)
print("\nFacility eligibility (lever 0):")
print(fac_elig_0)
print("\nAmbulance eligibility (lever 0):")
print(amb_elig_0)

---
## Step 1 — Build `pending_requests_df`
Mirrors `TransportC2.receive_request()`: assembles a flat DataFrame from individual request objects.

In [ ]:
def build_pending_df(requests: list) -> pd.DataFrame:
    rows = []
    for req in requests:
        rows.append({
            "req_id":              req.req_id,
            "casualty_id":         req.casualty_id,
            "triage":              req.triage,
            "issuing_facility_id": req.issuing_facility.loc_id,
            "injury_time":         req.injury_time,
            "t_issued":            req.t_issued,
            "needed_unit":         req.needed_unit,
            "status":              req.status,
        })
    return pd.DataFrame(rows)


pending_df = build_pending_df(ALL_REQUESTS)
print("pending_df:")
pending_df

---
## Step 2 — `fac_eligibility`
Reads `env.fac_eligibility_datasets[lever]` and adds column `eligible_facilities` to the requests.

Source: `src/policies/policies.py :: fac_eligibility`

In [ ]:
def fac_eligibility(env: Env, requests_df: pd.DataFrame, lever_value: int) -> pd.DataFrame:
    df = requests_df.copy()
    df["eligible_facilities"] = [[] for _ in range(len(df))]

    if df.empty:
        return df

    matrix = (env.fac_eligibility_datasets or {}).get(lever_value)
    if matrix is None or matrix.empty:
        return df

    def get_eligible(issuing_fac_id):
        if issuing_fac_id not in matrix.index:
            return []
        row = matrix.loc[issuing_fac_id]
        return [col for col in row.index if row[col] == 1]

    df["eligible_facilities"] = df["issuing_facility_id"].apply(get_eligible)
    return df


# ── test ──────────────────────────────────────────────────────────────────────
eligible_df = fac_eligibility(env, pending_df, lever_value=0)
print("After fac_eligibility:")
eligible_df[["req_id", "triage", "issuing_facility_id", "eligible_facilities"]]

In [ ]:
# ── edge case: unknown lever → all eligible_facilities empty ──────────────────
out_bad_lever = fac_eligibility(env, pending_df, lever_value=99)
assert out_bad_lever["eligible_facilities"].apply(len).sum() == 0

# ── edge case: empty request df ───────────────────────────────────────────────
out_empty = fac_eligibility(env, pd.DataFrame(columns=pending_df.columns), lever_value=0)
assert out_empty.empty

# ── edge case: issuing facility not in matrix index ───────────────────────────
unknown_req_df = pending_df.copy()
unknown_req_df.loc[0, "issuing_facility_id"] = "unknown_fac"
out_unknown = fac_eligibility(env, unknown_req_df, lever_value=0)
assert out_unknown.loc[0, "eligible_facilities"] == []

print("All edge-case tests for fac_eligibility passed.")

---
## Step 3 — `amb_eligibility`
Mirrors `fac_eligibility`: reads `env.amb_eligibility_datasets[lever]` and adds column
`eligible_ambulances` to each request row — a list of ambulance IDs that are permitted
to serve that request's issuing facility.

This keeps ambulance filtering outside the ILP, making the constraint explicit and testable.

In [ ]:
def amb_eligibility(
    env: Env,
    requests_df: pd.DataFrame,
    idle_ambulances: List[Ambulance],
    lever_value: int,
) -> pd.DataFrame:
    """Add 'eligible_ambulances' column: list of ambulance IDs allowed to serve each request."""
    df       = requests_df.copy()
    matrix   = (env.amb_eligibility_datasets or {}).get(lever_value)
    idle_ids = [a.amb_id for a in idle_ambulances]

    def get_eligible(origin: str) -> list:
        if matrix is None or matrix.empty:
            return idle_ids
        return [
            amb_id for amb_id in idle_ids
            if amb_id in matrix.index
            and origin in matrix.columns
            and matrix.loc[amb_id, origin] == 1
        ]

    df["eligible_ambulances"] = df["issuing_facility_id"].apply(get_eligible)
    return df


# ── test ──────────────────────────────────────────────────────────────────────
idle_ambs     = [amb_1, amb_2]
eligible_df_f = fac_eligibility(env, build_pending_df(ALL_REQUESTS), lever_value=0)
eligible_df_a = amb_eligibility(env, eligible_df_f, idle_ambs, lever_value=0)

print("After amb_eligibility:")
eligible_df_a[["req_id", "triage", "issuing_facility_id",
               "eligible_facilities", "eligible_ambulances"]]

In [ ]:
# ── edge cases ────────────────────────────────────────────────────────────────
# Unknown lever → falls back to all idle ambulances
out = amb_eligibility(env, eligible_df_f, idle_ambs, lever_value=99)
assert all(set(v) == {"amb_1", "amb_2"} for v in out["eligible_ambulances"])
print("edge case (unknown lever → all idle): OK")

# Empty ambulance list → all rows get empty list
out = amb_eligibility(env, eligible_df_f, [], lever_value=0)
assert out["eligible_ambulances"].apply(len).sum() == 0
print("edge case (no idle ambulances): OK")

# Restricted matrix: only amb_1 eligible
restricted = pd.DataFrame([[1,1,1,1,1],[0,0,0,0,0]],
                           index=["amb_1","amb_2"], columns=fac_ids)
env.amb_eligibility_datasets[1] = restricted
out = amb_eligibility(env, eligible_df_f, idle_ambs, lever_value=1)
assert all(v == ["amb_1"] for v in out["eligible_ambulances"])
print("edge case (restricted matrix — only amb_1): OK")
env.amb_eligibility_datasets.pop(1)  # restore

---
## Step 4 — `solve_ilp`
4-term weighted ILP (PuLP/CBC).

Variable `x[r, a, f] ∈ {0,1}` = 1 if request r is served by ambulance a to facility f.  
Variable `y[a, f] ∈ {0,1}` = 1 if ambulance a is dispatched to facility f (same-destination constraint).

Objective: **maximise** `Σ x[r,a,f] · score(r,a,f)`

```
score(r, a, f) = M
               + α · triage_value(r)          T1=3, T2=2, T3=1
               + β · sojourn(r)               now − t_issued
               − γ · transport_time(r, a, f)  d(amb→origin) + d(origin→facility)
               − δ · operation_time(r, f)     expected wait if joined now
```

**M** ensures any assignment beats leaving a seat empty.  
**y** constraints force all patients on the same ambulance to share one destination.

In [ ]:
# ── facility snapshot (mirrors snapshot_facilities) ───────────────────────────

def snapshot_facilities(facilities: List[Facility]) -> pd.DataFrame:
    """Return one row per facility, one column per service type.
    Cell = expected wait to join that unit; False if unit absent."""
    if not facilities:
        return pd.DataFrame()
    all_service_types = sorted({st for fac in facilities for st in fac.units})
    rows = []
    for fac in facilities:
        row = {"facility_id": fac.loc_id}
        for st in all_service_types:
            u = fac.units.get(st)
            row[st] = float(u.expected_wait_if_join_now()) if u else False
        rows.append(row)
    return pd.DataFrame(rows)


snap = snapshot_facilities(ALL_FACILITIES)
print("Facility snapshot (expected wait per unit):")
snap

In [ ]:
def solve_ilp(
    env: Env,
    requests_df: pd.DataFrame,
    idle_ambulances: List[Ambulance],
    distance_matrix: Optional[pd.DataFrame],
    alpha: float = 1.0,
    beta:  float = 0.0,
    gamma: float = 0.0,
    delta: float = 0.0,
    M:     float = 10_000.0,
    random_seed: int = 42,
) -> pd.DataFrame:
    """
    4-term ILP with same-destination constraint.

    Reads eligible_facilities and eligible_ambulances from requests_df
    (set by fac_eligibility and amb_eligibility respectively).
    """
    df = requests_df.copy()
    if df.empty:
        df["assigned_facility_id"] = None
        df["assigned_amb_id"] = None
        return df

    fac_snap  = snapshot_facilities(list(env.facility_by_id.values()))
    fac_index = fac_snap.set_index("facility_id") if not fac_snap.empty else pd.DataFrame()
    caps      = {a.amb_id: int(getattr(a, "capacity", 1)) for a in idle_ambulances}
    req_idx   = df.index.tolist()
    _TV       = {"T1": 3.0, "T2": 2.0, "T3": 1.0}

    prob        = LpProblem("ilp_4term", LpMaximize)
    x, y, costs = {}, {}, {}

    for idx, row in df.iterrows():
        eligible_facs = row.get("eligible_facilities") or []
        eligible_ambs = row.get("eligible_ambulances") or []
        origin        = row["issuing_facility_id"]
        unit_type     = row.get("needed_unit")
        triage_val    = _TV.get(str(row.get("triage", "")), 1.0)
        sojourn       = max(env.now() - float(row.get("t_issued", env.now())), 0.0)

        for amb_id in eligible_ambs:
            amb = env.amb_by_id.get(str(amb_id))
            if amb is None:
                continue

            amb_loc  = amb._loc_id(amb.location)
            d_pickup = (
                float(distance_matrix.loc[amb_loc, origin])
                if distance_matrix is not None
                   and amb_loc in distance_matrix.index
                   and origin  in distance_matrix.columns
                else 0.0
            )

            for fac_id in eligible_facs:
                d_transport = (
                    float(distance_matrix.loc[origin, fac_id])
                    if distance_matrix is not None
                       and origin in distance_matrix.index
                       and fac_id in distance_matrix.columns
                    else 0.0
                )

                ewt = 0.0
                if (
                    not fac_index.empty
                    and fac_id    in fac_index.index
                    and unit_type
                    and unit_type in fac_index.columns
                ):
                    val = fac_index.at[fac_id, unit_type]
                    if val is not False and not (isinstance(val, float) and pd.isna(val)):
                        ewt = float(val)

                key        = (idx, amb_id, fac_id)
                x[key]     = LpVariable(f"x_{idx}_{amb_id}_{fac_id}", cat="Binary")
                costs[key] = (
                    M
                    + alpha * triage_val
                    + beta  * sojourn
                    - gamma * (d_pickup + d_transport)
                    - delta * ewt
                )

                if (amb_id, fac_id) not in y:
                    y[(amb_id, fac_id)] = LpVariable(f"y_{amb_id}_{fac_id}", cat="Binary")

    if not x:
        df["assigned_facility_id"] = None
        df["assigned_amb_id"] = None
        return df

    # objective
    prob += lpSum(costs[k] * x[k] for k in x)

    # each request assigned at most once
    for idx in req_idx:
        vars_r = [x[k] for k in x if k[0] == idx]
        if vars_r:
            prob += lpSum(vars_r) <= 1

    # same-destination: x[r,a,f] <= cap(a) * y[a,f]  and  Σ_f y[a,f] <= 1
    for (amb_id, fac_id), yvar in y.items():
        vars_xaf = [x[k] for k in x if k[1] == amb_id and k[2] == fac_id]
        if vars_xaf:
            prob += lpSum(vars_xaf) <= caps.get(amb_id, 1) * yvar
    for amb_id in caps:
        vars_y = [y[k] for k in y if k[0] == amb_id]
        if vars_y:
            prob += lpSum(vars_y) <= 1

    # capacity: assignments per ambulance <= capacity
    for amb_id in caps:
        vars_a = [x[k] for k in x if k[1] == amb_id]
        if vars_a:
            prob += lpSum(vars_a) <= caps[amb_id]

    prob.solve(PULP_CBC_CMD(msg=0, options=[f"RandomS {random_seed}"]))

    af = {i: None for i in req_idx}
    aa = {i: None for i in req_idx}
    for (idx, amb_id, fac_id), var in x.items():
        if (value(var) or 0) > 0.5:
            af[idx] = fac_id
            aa[idx] = amb_id

    df["assigned_facility_id"] = [af[i] for i in req_idx]
    df["assigned_amb_id"]      = [aa[i] for i in req_idx]
    return df

In [ ]:
idle_ambs = [amb_1, amb_2]
elig_df   = amb_eligibility(env, eligible_df_f, idle_ambs, lever_value=0)

assigned_df = solve_ilp(
    env, elig_df, idle_ambs, distance_matrix,
    alpha=1.0, beta=0.0, gamma=0.0, delta=0.0,
)

print("Assignment result (triage only, α=1):")
assigned_df[["req_id", "triage", "issuing_facility_id",
             "assigned_facility_id", "assigned_amb_id"]]

In [ ]:
# ── Compare weight configurations ─────────────────────────────────────────────
CONFIGS = [
    dict(label="triage only",       alpha=1.0, beta=0.0,  gamma=0.0,  delta=0.0),
    dict(label="+ sojourn",         alpha=1.0, beta=1.0,  gamma=0.0,  delta=0.0),
    dict(label="+ transport",       alpha=1.0, beta=0.0,  gamma=1.0,  delta=0.0),
    dict(label="+ EWT",             alpha=1.0, beta=0.0,  gamma=0.0,  delta=1.0),
    dict(label="balanced",          alpha=1.0, beta=1.0,  gamma=0.5,  delta=0.5),
]

results = {}
for cfg in CONFIGS:
    res = solve_ilp(env, elig_df, idle_ambs, distance_matrix,
                    alpha=cfg["alpha"], beta=cfg["beta"],
                    gamma=cfg["gamma"], delta=cfg["delta"])
    results[cfg["label"]] = res[["req_id", "triage",
                                  "assigned_facility_id", "assigned_amb_id"]].copy()
    results[cfg["label"]].columns = ["req_id", "triage",
                                      f"dest_{cfg['label']}", f"amb_{cfg['label']}"]

comparison = results["triage only"]
for lbl in list(results.keys())[1:]:
    comparison = comparison.merge(results[lbl], on=["req_id", "triage"])

print("Assignment comparison across weight configurations:")
comparison

In [ ]:
# ── capacity constraint test ───────────────────────────────────────────────────
# amb_1 capacity=2, amb_2 capacity=2  →  max 4 requests assignable per cycle
w = compute_weights(eligible_df, lever_value=1, now=env.now())
res = solve_joint_ilp(env, eligible_df, idle_ambs, 0, distance_matrix, w)

for amb_id, grp in res.dropna(subset=["assigned_amb_id"]).groupby("assigned_amb_id"):
    cap = env.amb_by_id[amb_id].capacity
    n   = len(grp)
    assert n <= cap, f"{amb_id} over capacity: {n} > {cap}"
    print(f"{amb_id}: {n}/{cap} seats used")

print("Capacity constraint test passed.")

In [ ]:
# ── edge case: no idle ambulances ─────────────────────────────────────────────
w = compute_weights(eligible_df, lever_value=0, now=env.now())
res_no_ambs = solve_joint_ilp(env, eligible_df, [], 0, distance_matrix, w)
assert res_no_ambs["assigned_amb_id"].isna().all()

# ── edge case: no eligible facilities ─────────────────────────────────────────
df_no_elig = eligible_df.copy()
df_no_elig["eligible_facilities"] = [[] for _ in range(len(df_no_elig))]
res_no_facs = solve_joint_ilp(env, df_no_elig, idle_ambs, 0, distance_matrix, w)
assert res_no_facs["assigned_facility_id"].isna().all()

# ── edge case: no distance matrix ─────────────────────────────────────────────
res_no_dist = solve_joint_ilp(env, eligible_df, idle_ambs, 0, None, w)
n_assigned = res_no_dist["assigned_amb_id"].notna().sum()
print(f"Without distance matrix: {n_assigned} requests assigned (costs default to 0, still feasible)")

# ── edge case: amb_elig_lever with no matching dataset → all ambs eligible ────
res_no_ae = solve_joint_ilp(env, eligible_df, idle_ambs, 99, distance_matrix, w)
n_assigned_no_ae = res_no_ae["assigned_amb_id"].notna().sum()
print(f"With unknown amb_elig_lever (99): {n_assigned_no_ae} requests assigned")

print("All ILP edge-case tests passed.")

---
## Step 4 — `craft_plans`
Converts the assignment DataFrame into a `{ amb_id: [pickup_step, dropoff_step] }` dict.

Logic:
- Group assigned rows by ambulance
- Pick the issuing facility with the most rows as the **pickup** stop
- Truncate to ambulance capacity
- Pick the most common destination as the **dropoff** stop
- Resolve `Patient` objects from `pickup_fac.exit_queue` via `req_id`

Source: `src/policies/policies.py :: craft_plans`

In [ ]:
def craft_plans(
    env: Env,
    assigned_ambulances_df: pd.DataFrame,
) -> Dict[str, List[PlanStep]]:
    plans: Dict[str, List[PlanStep]] = {}

    if assigned_ambulances_df is None or assigned_ambulances_df.empty:
        return plans

    df = assigned_ambulances_df.copy()
    required = {"req_id", "issuing_facility_id", "assigned_facility_id", "assigned_amb_id"}
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"craft_plans missing columns: {missing}")

    # only work with actually assigned rows
    df = df.dropna(subset=["assigned_amb_id", "assigned_facility_id"])

    def req_map_for_facility(fac: Facility) -> dict:
        return {getattr(r, "req_id", None): r for r in getattr(fac, "exit_queue", [])}

    for amb_id, g in df.groupby("assigned_amb_id", sort=False):
        if amb_id is None:
            continue
        amb_id = str(amb_id)

        amb = env.amb_by_id.get(amb_id)
        if amb is None:
            continue

        # pickup = issuing facility with most cargo for this ambulance
        pickup_facility_id = g["issuing_facility_id"].value_counts().index[0]
        gg = g[g["issuing_facility_id"] == pickup_facility_id].copy()

        # truncate to capacity
        gg = gg.sample(frac=1, random_state=42).reset_index(drop=True)
        gg = gg.iloc[: min(int(amb.capacity), len(gg))]

        # dropoff = most common destination among selected rows
        dropoff_facility_id = gg["assigned_facility_id"].value_counts().index[0]

        pickup_fac  = env.facility_by_id.get(pickup_facility_id)
        dropoff_fac = env.facility_by_id.get(dropoff_facility_id)
        if pickup_fac is None or dropoff_fac is None:
            continue

        req_ids  = gg["req_id"].astype(str).tolist()
        rmap     = req_map_for_facility(pickup_fac)
        req_objs = [rmap.get(rid) for rid in req_ids]
        req_objs = [r for r in req_objs if r is not None and r.patient is not None]
        patients = [r.patient for r in req_objs]

        if not patients:
            continue

        for req in req_objs:
            req.status = "assigned"

        pickup_step  = PlanStep(facility=pickup_fac,  unload_patients=[],      load_patients=patients)
        dropoff_step = PlanStep(facility=dropoff_fac, unload_patients=patients, load_patients=[])
        plans[amb_id] = [pickup_step, dropoff_step]

    return plans


# reset request statuses before running
reset_scenario()

plans = craft_plans(env, assigned_df)

print("Plans crafted:")
for amb_id, steps in plans.items():
    pickup, dropoff = steps
    patient_ids = [p.patient_id for p in pickup.load_patients]
    triages     = [p.triage     for p in pickup.load_patients]
    print(f"  {amb_id}: {pickup.facility.loc_id} → {dropoff.facility.loc_id}  "
          f"patients={patient_ids}  triages={triages}")

In [ ]:
# ── validate plan structure ────────────────────────────────────────────────────
for amb_id, steps in plans.items():
    assert len(steps) == 2, f"{amb_id}: expected 2 steps"
    pickup, dropoff = steps
    assert len(pickup.unload_patients) == 0,  "pickup step must not unload"
    assert len(dropoff.load_patients) == 0,   "dropoff step must not load"
    assert pickup.load_patients == dropoff.unload_patients, "patient lists must match across steps"
    assert len(pickup.load_patients) <= env.amb_by_id[amb_id].capacity, f"{amb_id} over capacity"

print("Plan structure validation passed.")

In [ ]:
# ── edge case: all unassigned → empty plans ───────────────────────────────────
empty_assigned = assigned_df.copy()
empty_assigned["assigned_amb_id"] = None
empty_assigned["assigned_facility_id"] = None
plans_empty = craft_plans(env, empty_assigned)
assert plans_empty == {}
print("edge case (no assignments): OK")

# ── edge case: exit_queue empty → patient lookup fails silently ───────────────
reset_scenario()
ccp.exit_queue = []  # clear after resetting statuses
plans_no_queue = craft_plans(env, assigned_df)
for amb_id, steps in plans_no_queue.items():
    assert steps[0].facility is not ccp, "should not craft plan when exit_queue is empty"
reset_scenario()  # restore
print("edge case (empty exit_queue): OK")

# ── edge case: missing required column ────────────────────────────────────────
try:
    craft_plans(env, assigned_df.drop(columns=["assigned_facility_id"]))
    assert False, "should have raised ValueError"
except ValueError as e:
    print(f"edge case (missing column): caught ValueError — {e}")

---
## Step 5 — `dispatch_ambulances`
Attaches plans to ambulances, removes them from `idle_ambulances`, and marks the matching rows
in `pending_transport_requests_df` as `"assigned"`.

In the real simulation `amb.activate()` is also called here; we omit that since there is no salabim.

Source: `src/policies/policies.py :: dispatch_ambulances`

In [ ]:
def dispatch_ambulances(
    env: Env,
    idle_ambulances: List[Ambulance],
    pending_df: pd.DataFrame,
    plans_craft: Dict[str, List[PlanStep]],
) -> pd.DataFrame:
    """
    Returns the updated pending_df.
    Side effect: mutates idle_ambulances and attaches plan to each dispatched ambulance.
    """
    pending_df = pending_df.copy()

    for amb_id, plan in (plans_craft or {}).items():
        amb = env.amb_by_id.get(amb_id)
        if amb is None:
            continue
        if amb not in idle_ambulances:
            continue

        amb.plan = plan
        idle_ambulances.remove(amb)

        loaded_patient_ids = {p.patient_id for step in plan for p in step.load_patients}
        if loaded_patient_ids:
            mask = (
                (pending_df["status"] == "pending")
                & (pending_df["casualty_id"].astype(str).isin(
                    {str(x) for x in loaded_patient_ids}
                ))
            )
            pending_df.loc[mask, "status"] = "assigned"

    return pending_df


# ── fresh run ─────────────────────────────────────────────────────────────────
reset_scenario()

idle_ambs_live  = [amb_1, amb_2]
pending_df_live = build_pending_df(ALL_REQUESTS)
elig            = fac_eligibility(env, pending_df_live, lever_value=0)
w               = compute_weights(elig, lever_value=1, now=env.now())
ass             = solve_joint_ilp(env, elig, idle_ambs_live, 0, distance_matrix, w)
plans_live      = craft_plans(env, ass)

reset_scenario()

pending_df_after = dispatch_ambulances(env, idle_ambs_live, pending_df_live, plans_live)

print("Remaining idle ambulances:", [a.amb_id for a in idle_ambs_live])
print("\nPending df after dispatch:")
pending_df_after[["req_id", "casualty_id", "triage", "status"]]

In [ ]:
# ── validate dispatch results ─────────────────────────────────────────────────
dispatched_ambs = [a for a in ALL_AMBULANCES if a not in idle_ambs_live]
print("Dispatched ambulances:")
for a in dispatched_ambs:
    if a.plan:
        pickup, dropoff = a.plan
        patients = [p.patient_id for p in pickup.load_patients]
        print(f"  {a.amb_id}: {pickup.facility.loc_id} → {dropoff.facility.loc_id}  patients={patients}")

n_assigned = (pending_df_after["status"] == "assigned").sum()
n_pending  = (pending_df_after["status"] == "pending").sum()
print(f"\nRequests assigned: {n_assigned}  |  still pending: {n_pending}")

for a in dispatched_ambs:
    assert a not in idle_ambs_live, f"{a.amb_id} should have been removed from idle list"
print("dispatch validation passed.")

In [ ]:
# ── edge case: ambulance not in idle list → skipped silently ──────────────────
idle_fresh    = []
pending_fresh = build_pending_df(ALL_REQUESTS)
result_no_idle = dispatch_ambulances(env, idle_fresh, pending_fresh, plans_live)
assert (result_no_idle["status"] == "pending").all(), "no statuses should change when idle list is empty"

# ── edge case: empty plans dict → nothing changes ─────────────────────────────
idle_full  = [amb_1, amb_2]
result_no_plans = dispatch_ambulances(env, idle_full, pending_fresh, {})
assert len(idle_full) == 2, "idle list should be unchanged"
assert (result_no_plans["status"] == "pending").all()

print("dispatch edge-case tests passed.")

---
## End-to-End Pipeline
Full single-cycle run mirroring `TransportC2.process()`.

In [ ]:
def run_c2_cycle(
    env: Env,
    requests: List[TransportRequest],
    idle_ambulances: List[Ambulance],
    lever_fac_elig: int = 0,
    lever_joint: int = 1,
    lever_amb_elig: int = 0,
) -> dict:
    """One full C2 dispatch cycle."""
    pending_df = build_pending_df(requests)
    requests_df = pending_df[pending_df["status"] == "pending"].copy()

    elig_df     = fac_eligibility(env, requests_df, lever_value=lever_fac_elig)
    weights     = compute_weights(elig_df, lever_value=lever_joint, now=env.now())
    assigned_df = solve_joint_ilp(env, elig_df, idle_ambulances, lever_amb_elig, env.distance_matrix, weights)
    plans       = craft_plans(env, assigned_df)
    final_df    = dispatch_ambulances(env, idle_ambulances, pending_df, plans)

    return {"pending_df": final_df, "assigned_df": assigned_df, "plans": plans}


# fresh state
reset_scenario()
idle = [amb_1, amb_2]

result = run_c2_cycle(env, ALL_REQUESTS, idle, lever_fac_elig=0, lever_joint=1)

print("=== E2E result ===")
print("Plans:")
for amb_id, steps in result["plans"].items():
    patients = [pt.patient_id for pt in steps[0].load_patients]
    triages  = [pt.triage     for pt in steps[0].load_patients]
    print(f"  {amb_id}: {steps[0].facility.loc_id} → {steps[1].facility.loc_id}  "
          f"patients={patients}  triages={triages}")

print("\nFinal request statuses:")
result["pending_df"][["req_id", "triage", "issuing_facility_id", "status"]]

---
## Scenario Comparisons
Run the same requests under all three priority levers and compare which patients get served and where.

In [ ]:
LEVER_LABELS = {0: "FIFO", 1: "Triage", 2: "Triage+FIFO"}
rows = []

for lever in [0, 1, 2]:
    reset_scenario()
    idle = [amb_1, amb_2]

    r = run_c2_cycle(env, ALL_REQUESTS, idle, lever_fac_elig=0, lever_joint=lever)
    n_assigned = (r["pending_df"]["status"] == "assigned").sum()
    n_pending  = (r["pending_df"]["status"] == "pending").sum()

    for amb_id, steps in r["plans"].items():
        patients = [pt.patient_id for pt in steps[0].load_patients]
        triages  = [pt.triage     for pt in steps[0].load_patients]
        rows.append({
            "lever":      LEVER_LABELS[lever],
            "ambulance":  amb_id,
            "pickup":     steps[0].facility.loc_id,
            "dropoff":    steps[1].facility.loc_id,
            "patients":   patients,
            "triages":    triages,
            "n_assigned": n_assigned,
            "n_pending":  n_pending,
        })

pd.DataFrame(rows)

---
## Pluggable Assignment Policies

The assignment step has a well-defined **interface contract**:

```
(env, requests_df, idle_ambulances, amb_elig_lever, distance_matrix) → DataFrame
```

The output DataFrame must extend `requests_df` with two extra columns:

| column | type | meaning |
|---|---|---|
| `assigned_facility_id` | `str \| None` | destination facility, `None` = unassigned |
| `assigned_amb_id` | `str \| None` | assigned ambulance, `None` = unassigned |

Any function with this signature can be passed to `run_c2_cycle_with_policy`.
Priority logic, weights, and tie-breaking are the **policy's own responsibility** — the runner just passes data in and reads assignments out.

In [ ]:
from typing import Callable

# Type alias for an assignment policy function
AssignmentPolicy = Callable[
    [Env, pd.DataFrame, List[Ambulance], int, Optional[pd.DataFrame]],
    pd.DataFrame,
]


def run_c2_cycle_with_policy(
    env: Env,
    requests: List[TransportRequest],
    idle_ambulances: List[Ambulance],
    assignment_policy: AssignmentPolicy,
    lever_fac_elig: int = 0,
    lever_amb_elig: int = 0,
) -> dict:
    """Generic C2 cycle — swap any assignment policy in."""
    pending_df  = build_pending_df(requests)
    requests_df = pending_df[pending_df["status"] == "pending"].copy()

    elig_df     = fac_eligibility(env, requests_df, lever_value=lever_fac_elig)
    assigned_df = assignment_policy(env, elig_df, idle_ambulances, lever_amb_elig, env.distance_matrix)

    plans    = craft_plans(env, assigned_df)
    final_df = dispatch_ambulances(env, idle_ambulances, pending_df, plans)

    return {"pending_df": final_df, "assigned_df": assigned_df, "plans": plans}


print("run_c2_cycle_with_policy defined.")

In [ ]:
def make_ilp_orig(lever_weights: int = 1) -> AssignmentPolicy:
    """
    Wraps solve_joint_ilp as a pluggable policy.
    lever_weights: 0=FIFO, 1=triage, 2=triage+FIFO
    """
    def solve(env, requests_df, idle_ambulances, amb_elig_lever, distance_matrix):
        weights = compute_weights(requests_df, lever_value=lever_weights, now=env.now())
        return solve_joint_ilp(env, requests_df, idle_ambulances, amb_elig_lever, distance_matrix, weights)

    solve.__name__ = f"ilp_orig(lever={lever_weights})"
    return solve


print("make_ilp_orig defined.")

### Myopic FIFO Policy

Processes requests one at a time in **FIFO order** (oldest `t_issued` first).  
For each request it greedily picks:
- the **closest eligible idle ambulance** (by distance from ambulance's current location to the issuing facility)
- the **closest eligible destination facility** (by distance from the issuing facility)

An ambulance stays in the available pool until its capacity is exhausted.  
No global optimisation — each decision is made locally without considering future requests.

In [ ]:
def assign_myopic_fifo(
    env: Env,
    requests_df: pd.DataFrame,
    idle_ambulances: List[Ambulance],
    amb_elig_lever: int,
    distance_matrix: Optional[pd.DataFrame],
) -> pd.DataFrame:
    """
    Myopic FIFO assignment policy.

    For each request (oldest first):
      1. Collect idle ambulances with remaining capacity that are eligible
         to serve the issuing facility.
      2. Pick the one closest to the issuing facility.
      3. Among eligible destination facilities, pick the one closest to
         the issuing facility.
    """
    df = requests_df.copy()
    df["assigned_facility_id"] = None
    df["assigned_amb_id"]      = None

    if df.empty or not idle_ambulances:
        return df

    ae = (env.amb_eligibility_datasets or {}).get(amb_elig_lever)
    remaining_cap = {a.amb_id: a.capacity for a in idle_ambulances}

    def get_dist(from_id: str, to_id: str) -> float:
        if distance_matrix is None:
            return 0.0
        if from_id in distance_matrix.index and to_id in distance_matrix.columns:
            return float(distance_matrix.loc[from_id, to_id])
        return float("inf")

    for idx in df.sort_values("t_issued").index:
        row           = df.loc[idx]
        origin        = row["issuing_facility_id"]
        eligible_facs = row.get("eligible_facilities") or []

        if not eligible_facs:
            continue

        eligible_ambs = []
        for a in idle_ambulances:
            if remaining_cap.get(a.amb_id, 0) <= 0:
                continue
            if ae is not None and not ae.empty:
                if (a.amb_id not in ae.index
                        or origin not in ae.columns
                        or ae.loc[a.amb_id, origin] != 1):
                    continue
            eligible_ambs.append(a)

        if not eligible_ambs:
            continue

        best_amb    = min(eligible_ambs, key=lambda a: get_dist(a._loc_id(a.location), origin))
        best_fac_id = min(eligible_facs, key=lambda f: get_dist(origin, f))

        df.at[idx, "assigned_amb_id"]      = best_amb.amb_id
        df.at[idx, "assigned_facility_id"] = best_fac_id
        remaining_cap[best_amb.amb_id]    -= 1

    return df


print("assign_myopic_fifo defined.")

In [ ]:
# ── run myopic policy ─────────────────────────────────────────────────────────
reset_scenario()
idle = [amb_1, amb_2]

result_myopic = run_c2_cycle_with_policy(
    env, ALL_REQUESTS, idle,
    assignment_policy=assign_myopic_fifo,
    lever_fac_elig=0,
    lever_amb_elig=0,
)

print("=== Myopic FIFO result ===")
for amb_id, steps in result_myopic["plans"].items():
    patients = [pt.patient_id for pt in steps[0].load_patients]
    triages  = [pt.triage     for pt in steps[0].load_patients]
    print(f"  {amb_id}: {steps[0].facility.loc_id} → {steps[1].facility.loc_id}  "
          f"patients={patients}  triages={triages}")

print()
result_myopic["assigned_df"][[
    "req_id", "triage", "t_issued", "issuing_facility_id",
    "eligible_facilities", "assigned_facility_id", "assigned_amb_id",
]]

In [ ]:
# ── edge cases ────────────────────────────────────────────────────────────────

# 1. No eligible facilities → all unassigned
df_no_elig = eligible_df.copy()
df_no_elig["eligible_facilities"] = [[] for _ in range(len(df_no_elig))]
res = assign_myopic_fifo(env, df_no_elig, [amb_1, amb_2], 0, distance_matrix)
assert res["assigned_amb_id"].isna().all()
print("edge case (no eligible facs): OK")

# 2. No ambulances → all unassigned
res = assign_myopic_fifo(env, eligible_df, [], 0, distance_matrix)
assert res["assigned_amb_id"].isna().all()
print("edge case (no ambulances): OK")

# 3. Capacity exhausted mid-batch
for req in ALL_REQUESTS:
    req.status = "pending"
res = assign_myopic_fifo(env, eligible_df, [amb_1], 0, distance_matrix)
n = res["assigned_amb_id"].notna().sum()
assert n <= amb_1.capacity, f"exceeded capacity: {n} > {amb_1.capacity}"
print(f"edge case (single cap-2 ambulance): {n} request(s) assigned — OK")

# 4. Ordering: oldest requests should be processed first
res = assign_myopic_fifo(env, eligible_df, [amb_1], 0, distance_matrix)
assigned_idx = res.dropna(subset=["assigned_amb_id"])
oldest_2_t = eligible_df["t_issued"].nsmallest(amb_1.capacity).max()
assert (assigned_idx["t_issued"] <= oldest_2_t).all(), "oldest requests should win"
print("edge case (FIFO ordering): OK")

### ILP vs Myopic — Side-by-Side Comparison

Both policies are run on the same scenario so you can see where they diverge.  
Key things to look for:
- Which requests get left unassigned under each policy?
- Does the myopic "closest ambulance + closest facility" choice differ from the ILP's globally optimal one?
- Does FIFO ordering cause the myopic policy to waste ambulance capacity on low-priority patients?

In [ ]:
def fresh_run(policy_fn, label):
    reset_scenario()
    idle = [amb_1, amb_2]

    r = run_c2_cycle_with_policy(
        env, ALL_REQUESTS, idle,
        assignment_policy=policy_fn,
        lever_fac_elig=0,
        lever_amb_elig=0,
    )
    adf = r["assigned_df"][["req_id", "triage", "t_issued",
                             "assigned_facility_id", "assigned_amb_id"]].copy()
    adf.insert(0, "policy", label)
    return adf


ilp_fifo   = fresh_run(make_ilp_orig(lever_weights=0), "ILP-FIFO")
ilp_triage = fresh_run(make_ilp_orig(lever_weights=1), "ILP-Triage")
myopic     = fresh_run(assign_myopic_fifo,              "Myopic-FIFO")

comparison = pd.concat([ilp_fifo, ilp_triage, myopic], ignore_index=True)

def fmt(row):
    if pd.isna(row["assigned_amb_id"]):
        return "—"
    return f"{row['assigned_amb_id']} → {row['assigned_facility_id']}"

comparison["result"] = comparison.apply(fmt, axis=1)
pivot = comparison.pivot(index=["req_id","triage","t_issued"], columns="policy", values="result")
pivot.columns.name = None
pivot.reset_index()

### Adding Your Own Policy

Copy this template — the only requirement is matching the function signature:

```python
def assign_my_policy(
    env: Env,
    requests_df: pd.DataFrame,       # has 'eligible_facilities' column from fac_eligibility
    idle_ambulances: List[Ambulance],
    amb_elig_lever: int,
    distance_matrix: Optional[pd.DataFrame],
    weights: dict,                    # {df_index: float} — use or ignore
) -> pd.DataFrame:
    df = requests_df.copy()
    df["assigned_facility_id"] = None  # required output column
    df["assigned_amb_id"]      = None  # required output column

    # ... your logic here ...

    return df
```

Then plug it in:

```python
result = run_c2_cycle_with_policy(
    env, ALL_REQUESTS, idle,
    assignment_policy=assign_my_policy,
    lever_fac_elig=0,
    lever_amb_elig=0,
)
```

---
### 4-Term Weighted ILP Policy

A factory that returns an `AssignmentPolicy` with an explicit 4-term objective:

```
score(r, a, f) = M
               + α · triage_value(r)          triage priority   (T1=3, T2=2, T3=1)
               + β · sojourn(r)               time waiting since t_issued
               − γ · transport_time(r, a, f)  d(amb → ccp) + d(ccp → facility)
               − δ · operation_time(r, f)     expected wait if joined now at facility
```

`M` ensures any assignment always scores higher than leaving a seat empty.  
Splitting transport and operation time gives independent control over each penalty.

In [ ]:
def make_ilp_4term(
    alpha: float = 1.0,   # triage priority weight
    beta:  float = 0.1,   # sojourn time weight
    gamma: float = 0.05,  # transport time weight  (d_pickup + d_transport)
    delta: float = 0.05,  # operation time weight  (EWT at destination)
    M:     float = 10_000.0,
) -> AssignmentPolicy:
    """
    Factory returning a joint ILP policy with a 4-term weighted objective.

        score(r, a, f) = M
                       + alpha * triage_value(r)
                       + beta  * sojourn(r)
                       - gamma * transport_time(r, a, f)
                       - delta * operation_time(r, f)
    """
    _TRIAGE_VAL = {"T1": 3.0, "T2": 2.0, "T3": 1.0}

    def solve(
        env: Env,
        requests_df: pd.DataFrame,
        idle_ambulances: List[Ambulance],
        amb_elig_lever: int,
        distance_matrix: Optional[pd.DataFrame],
    ) -> pd.DataFrame:
        df = requests_df.copy()
        if df.empty:
            df["assigned_facility_id"] = None
            df["assigned_amb_id"] = None
            return df

        fac_snap  = snapshot_facilities(list(env.facility_by_id.values()))
        fac_index = fac_snap.set_index("facility_id") if not fac_snap.empty else pd.DataFrame()

        ae         = (env.amb_eligibility_datasets or {}).get(amb_elig_lever)
        idle_ids   = [a.amb_id for a in idle_ambulances]
        req_indices = df.index.tolist()

        prob  = LpProblem("joint_dispatch_4term", LpMaximize)
        x     = {}
        costs = {}

        for idx, row in df.iterrows():
            eligible_facs = row.get("eligible_facilities") or []
            origin        = row["issuing_facility_id"]
            unit_type     = row.get("needed_unit")

            triage_val = _TRIAGE_VAL.get(str(row.get("triage", "")), 1.0)
            sojourn    = max(env.now() - float(row.get("t_issued", env.now())), 0.0)

            for amb_id in idle_ids:
                amb = env.amb_by_id.get(str(amb_id))
                if amb is None:
                    continue
                if ae is not None and not ae.empty:
                    if amb_id not in ae.index or origin not in ae.columns or ae.loc[amb_id, origin] != 1:
                        continue

                amb_loc  = amb._loc_id(amb.location)
                d_pickup = (
                    float(distance_matrix.loc[amb_loc, origin])
                    if distance_matrix is not None
                       and amb_loc in distance_matrix.index
                       and origin  in distance_matrix.columns
                    else 0.0
                )

                for fac_id in eligible_facs:
                    d_transport = (
                        float(distance_matrix.loc[origin, fac_id])
                        if distance_matrix is not None
                           and origin in distance_matrix.index
                           and fac_id in distance_matrix.columns
                        else 0.0
                    )

                    ewt = 0.0
                    if (
                        not fac_index.empty
                        and fac_id    in fac_index.index
                        and unit_type
                        and unit_type in fac_index.columns
                    ):
                        val = fac_index.at[fac_id, unit_type]
                        if val is not False and not (isinstance(val, float) and pd.isna(val)):
                            ewt = float(val)

                    transport_time = d_pickup + d_transport
                    operation_time = ewt

                    key = (idx, amb_id, fac_id)
                    x[key]     = LpVariable(f"x4_{idx}_{amb_id}_{fac_id}", cat="Binary")
                    costs[key] = (
                        M
                        + alpha * triage_val
                        + beta  * sojourn
                        - gamma * transport_time
                        - delta * operation_time
                    )

        if not x:
            df["assigned_facility_id"] = None
            df["assigned_amb_id"] = None
            return df

        prob += lpSum(costs[k] * x[k] for k in x)

        for idx in req_indices:
            vars_r = [x[k] for k in x if k[0] == idx]
            if vars_r:
                prob += lpSum(vars_r) <= 1

        for amb_id in idle_ids:
            vars_a = [x[k] for k in x if k[1] == amb_id]
            if vars_a:
                amb = env.amb_by_id.get(str(amb_id))
                cap = int(getattr(amb, "capacity", 1)) if amb else 1
                prob += lpSum(vars_a) <= cap

        prob.solve(PULP_CBC_CMD(msg=0))

        assigned_fac = {idx: None for idx in req_indices}
        assigned_amb = {idx: None for idx in req_indices}
        for (idx, amb_id, fac_id), var in x.items():
            if (value(var) or 0) > 0.5:
                assigned_fac[idx] = fac_id
                assigned_amb[idx] = amb_id

        df["assigned_facility_id"] = [assigned_fac[i] for i in req_indices]
        df["assigned_amb_id"]      = [assigned_amb[i] for i in req_indices]
        return df

    solve.__name__ = f"ilp_4term(α={alpha},β={beta},γ={gamma},δ={delta})"
    return solve


print("make_ilp_4term defined.")

In [ ]:
# ── Compare policies side-by-side ─────────────────────────────────────────────
policies = [
    (make_ilp_orig(lever_weights=1),                               "ILP-orig (triage)"),
    (make_ilp_orig(lever_weights=0),                               "ILP-orig (FIFO)"),
    (make_ilp_4term(alpha=1.0, beta=0.1, gamma=0.05, delta=0.05), "4-term (balanced)"),
    (make_ilp_4term(alpha=1.0, beta=0.0, gamma=0.0,  delta=1.0),  "4-term (EWT-heavy)"),
    (make_ilp_4term(alpha=1.0, beta=0.0, gamma=1.0,  delta=0.0),  "4-term (transport-heavy)"),
    (assign_myopic_fifo,                                           "Myopic-FIFO"),
]

frames = []
for policy_fn, label in policies:
    reset_scenario()
    idle = [amb_1, amb_2]
    r = run_c2_cycle_with_policy(
        env, ALL_REQUESTS, idle,
        assignment_policy=policy_fn,
        lever_fac_elig=0,
        lever_amb_elig=0,
    )
    adf = r["assigned_df"][["req_id", "triage", "t_issued",
                             "assigned_facility_id", "assigned_amb_id"]].copy()
    adf["result"] = adf.apply(
        lambda row: "—" if pd.isna(row["assigned_amb_id"])
                    else f"{row['assigned_amb_id']} → {row['assigned_facility_id']}",
        axis=1,
    )
    adf.insert(0, "policy", label)
    frames.append(adf[["policy", "req_id", "triage", "t_issued", "result"]])

comparison = pd.concat(frames, ignore_index=True)
pivot = comparison.pivot(
    index=["req_id", "triage", "t_issued"],
    columns="policy",
    values="result",
)
pivot.columns.name = None
pivot.reset_index()